# 2 · Quem são os beneficiários dos rendimentos da dívida?

**Pergunta:** quando a União paga juros, para quem vai esse rendimento?

**Do que há evidência pública:**
- composição do estoque da DPMFi **por grupo de detentor** (Anexo 2.7 do RMD, série desde 2007);
- estimativa do **fluxo de juros por grupo** (juros nominais do ano × participação no estoque).

**O que NÃO é público:** nome por nome / instituição por instituição. O Requerimento
696/2019 (Câmara dos Deputados), que pedia essa lista ao STN, foi respondido com
**negativa por sigilo bancário**. Qualquer "banqueiro X recebeu Y bilhões" publicado
em rede social, sem fonte primária, é extrapolação.


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
dz = ind.detentores_anuais()
part = dz.pivot_table(index="data", columns="detentor", values="participacao") * 100
part.tail(3).round(1)

detentor,Fundos de Investimento,Governo,Instituições Financeiras,Não-residentes,Outros,Previdência,Seguradoras,Tesouro Direto
data,,,,,,,,
2023-12-31,23.50,3.80,29.70,9.50,4.50,23.00,4.00,2.00
2024-12-31,21.70,3.40,29.50,10.20,5.10,23.90,4.00,2.30
2025-12-31,20.80,2.90,32.90,10.30,4.30,22.80,3.50,2.60


In [3]:
ordem = ["Instituições Financeiras", "Fundos de Investimento", "Previdência",
         "Não-residentes", "Tesouro Direto", "Seguradoras", "Governo", "Outros"]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(part.index, [part[c].to_numpy() for c in ordem],
             labels=ordem, colors=[CORES_DETENTOR[c] for c in ordem], alpha=0.85)
ax.set_ylabel("% do estoque DPMFi")
ax.set_ylim(0, 100)
ax.set_title("Quem detém a dívida mobiliária federal em mercado (2007–2026)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_detentores_area")

fig -> viz/figs/02_detentores_area.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_area.png')

In [4]:
dez = dz[dz.data == dz.data.max()].sort_values("valor_bi", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(dez.detentor, dez.valor_bi, color=[CORES_DETENTOR[c] for c in dez.detentor])
for i, (v, pr) in enumerate(zip(dez.valor_bi, dez.participacao)):
    ax.text(v + 30, i, f"R$ {v:,.0f} bi ({pr:.1%})", va="center", fontsize=9)
ax.set_xlabel("R$ bilhões (estoque em dezembro do último ano completo)")
ax.set_xlim(0, dez.valor_bi.max() * 1.32)
ax.set_title("Estoque da DPMFi por detentor (dezembro)")
salvar(fig, "02_detentores_estoque")

fig -> viz/figs/02_detentores_estoque.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_estoque.png')

In [5]:
jf = ind.juros_por_detentor()
jf = jf[jf.ano < 2026]
conv = jf.pivot_table(index="ano", columns="detentor", values="juros_estimado_bi")
print("Juros nominais federais por ano (R$ bi):")
v = ind.juros_anuais()
ref = v[v.ano < 2026].set_index("ano")["juros_mi"] / 1000
print(ref.tail(12).round(0).to_string())

Juros nominais federais por ano (R$ bi):
ano
2014     311.00
2015     502.00
2016     407.00
2017     401.00
2018     379.00
2019     367.00
2020     312.00
2021     448.00
2022     586.00
2023     718.00
2024     950.00
2025   1,008.00


In [6]:
conv = conv.dropna(axis=1, how="all")
fig, ax = plt.subplots(figsize=(11, 5.5))
b = conv.loc[2015:]
ax.stackplot(b.index, [b[c].to_numpy() for c in b.columns],
             labels=list(b.columns),
             colors=[CORES_DETENTOR.get(c, "#999999") for c in b.columns], alpha=0.85)
ax.set_ylabel("R$ bilhões/ano (estimado)")
ax.set_title("Fluxo de juros estimado por grupo detentor (participação × juros nominais)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_juros_por_detentor")

fig -> viz/figs/02_juros_por_detentor.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_juros_por_detentor.png')

**Premissa do gráfico acima:** participação do estoque em **dezembro** de cada
ano × juros nominais do ano (SGS 4616). Não é dado oficial de fluxo por detentor —
a União não publica quem recebeu; é a **melhor aproximação possível com dados
públicos**, boa para o tamanho relativo dos beneficiários.

**Achados verificáveis (P2).**

- O **sistema financeiro** — Instituições Financeiras + Fundos de Investimento —
  detém cerca de metade do estoque e é o maior beneficiário estimado do fluxo
  de juros (só as instituições financeiras: ~R$ 330 bi no ano em 2025).
- **Previdência** (fundos de pensão patronais, incluindo planos de servidores e
  trabalhadores privados) é o 2º grupo: ~22–24%.
- **Pessoas físicas** (Tesouro Direto) têm participação pequena: ~3% — o padrão
  de beneficiário é **institucional**, não pulverizado como o discurso do
  "Tesouro Direto democratizou" sugere.
- **Não-residentes** oscilaram entre ~9% e ~14% da carteira média: parcela
  estruturalmente estrangeira, sensível a crises.

**Limitação:** saber *quais* instituições, dentro de cada grupo, detêm quanto
não é publicável (sigilo; Req. 696/2019 negado). O produto deve explicitar
isso — concentração intra-grupo é lacuna informada, não escondida.